# 08 — Model Comparison & Final Dashboard

We've covered a lot of ground:

- Data loading, panel completion, train/test split (notebook 01)
- Feature engineering with leakage-safe lags & rolling features (02)
- LightGBM recursive forecaster (03)
- Recursive vs direct vs purely causal strategies (04)
- Hyperparameter optimization with grid search & Optuna (05)
- Prophet for probabilistic forecasts and decomposition (06)
- Explainability with SHAP (07)

We'll fit four model variants on the
same data, score them on the same holdout, and assemble a comparison
dashboard.

| Model | Type | Probabilistic? |
|---|---|---|
| LightGBM Recursive | global, tree-boosted, one-step + roll-out | No |
| LightGBM Direct | global, one model per horizon | No |
| LightGBM Quantile | per-quantile boosters | **Yes** |
| Prophet | per-key, additive | **Yes** |


In [1]:
# === Colab / local setup ====================================================
# 1. Install dependencies (uncomment the pip line on first Colab run).
# !pip install -q lightgbm==4.* prophet plotly optuna shap pandas numpy scikit-learn pyarrow

# 2. Make the `utils` package importable. Two options:
#    (a) Notebook is sitting next to a `utils/` folder (recommended).
#    (b) The package is uploaded as a zip; unzip it and `sys.path.append(...)`.
import os, sys
HERE = os.path.dirname(os.path.abspath("__file__"))  # may be empty in Colab
for cand in [".", "..", "/content", "/content/ml_forecasting_tutorial"]:
    if os.path.isdir(os.path.join(cand, "utils")):
        sys.path.insert(0, cand)
        break

# 3. Standard imports for every notebook.
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "colab"   # works in Colab + Jupyter

# 4. Tell pandas to display nicely.
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


In [2]:
# === Dataset configuration ==================================================
# EDIT THIS CELL TO POINT AT YOUR DATASET
#DATA_PATH    = "/content/sales.csv"             # path to your Kaggle CSV / parquet
#DATE_COL     = "date"                           # date column
#TARGET_COL   = "sales"                          # target / forecast column
#KEY_COLS     = ["store", "item"]                # columns identifying a unique series
#EXCLUDE_COLS = []
#FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
#HOLDOUT_DAYS = 28                               # length of test horizon

DATA_PATH    = "./dataset/m5/m5_tiny.csv"             # path to your CSV / parquet
DATE_COL     = "date"                              # date column
TARGET_COL   = "sales"                          # target / forecast column
KEY_COLS     = ['item_id',
                'dept_id',
                'cat_id',
                'store_id',
                'state_id']                           # columns identifying a unique series
EXCLUDE_COLS = []
FREQ         = "D"                              # 'D'=daily, 'W'=weekly, 'M'=monthly
HOLDOUT_DAYS = 28                               # length of test horizon

## 1. Load data and prepare features

In [3]:
from utils.data_utils import load_forecasting_data, complete_panel, time_based_split
from utils.feature_engineering import FeatureEngineer

raw = load_forecasting_data(DATA_PATH, DATE_COL, TARGET_COL, KEY_COLS)
panel = complete_panel(raw, DATE_COL, KEY_COLS, TARGET_COL, freq=FREQ, fill_value=0.0)
cutoff = panel[DATE_COL].max() - pd.Timedelta(days=HOLDOUT_DAYS - 1)
train_df, test_df = time_based_split(panel, DATE_COL, cutoff=cutoff)

fe = FeatureEngineer(date_col=DATE_COL, target_col=TARGET_COL, key_cols=KEY_COLS,
                     lags=[1, 7, 14, 28], rolling_windows=[7, 28], ewm_halflives=[7.0, 28.0])
print(f"train: {len(train_df):,}   test: {len(test_df):,}   horizon: {HOLDOUT_DAYS} days")


train: 220,751   test: 14,000   horizon: 28 days


## 2. Fit the four candidate models

In [4]:
from utils.lgbm_forecaster import (
    RecursiveForecaster, DirectMultiHorizonForecaster, QuantileForecaster, DEFAULT_PARAMS
)
from utils.prophet_forecaster import MultiKeyProphet

# --- Model A: Recursive LightGBM -------------------------------------------
rec = RecursiveForecaster(feature_engineer=fe, params=DEFAULT_PARAMS,
                          num_boost_round=1500, early_stopping_rounds=100)
rec.fit(train_df, valid_df=test_df)

# --- Model B: Direct LightGBM ----------------------------------------------
direct = DirectMultiHorizonForecaster(feature_engineer=fe, horizons=list(range(1, HOLDOUT_DAYS + 1)),
                                      params=DEFAULT_PARAMS, num_boost_round=800,
                                      early_stopping_rounds=80)
direct.fit(train_df, valid_df=test_df)

# --- Model C: Quantile LightGBM (probabilistic) ----------------------------
quant = QuantileForecaster(feature_engineer=fe, quantiles=(0.1, 0.5, 0.9),
                           params=DEFAULT_PARAMS, num_boost_round=800,
                           early_stopping_rounds=80)
quant.fit(train_df, valid_df=test_df)

# --- Model D: Prophet (per-key, slow on many keys; sample if needed) -------
# To keep runtime sensible, restrict Prophet to the top-N keys; the others
# will be excluded from Prophet's row of the comparison.
top_keys = (train_df.groupby(KEY_COLS)[TARGET_COL].sum()
            .sort_values(ascending=False).head(20).index)
mask_train = train_df.set_index(KEY_COLS).index.isin(top_keys)
mask_test  = test_df.set_index(KEY_COLS).index.isin(top_keys)

prophet = MultiKeyProphet(date_col=DATE_COL, target_col=TARGET_COL, key_cols=KEY_COLS,
                          prophet_kwargs=dict(yearly_seasonality=True, weekly_seasonality=True,
                                              seasonality_mode="multiplicative",
                                              interval_width=0.80),
                          country_holidays="US")
prophet.fit(train_df[mask_train])
print("all four models fitted.")


06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1] done processing
06:00:55 - cmdstanpy - INFO - Chain [1] start processing
06:00:55 - cmdstanpy - INFO - Chain [1]

all four models fitted.


## 3. Generate predictions on the holdout

In [7]:
import pandas as pd

# A. Recursive
horizon_dates = sorted(test_df[DATE_COL].unique())
# Dataset Specific features
holiday_cols = [c for c in train_df.columns if c.startswith("event_")]
snap_cols = [c for c in train_df.columns if c.startswith("snap_")]
other_future_known_cols = ['sell_price']
exo_future = test_df[[DATE_COL] + KEY_COLS + holiday_cols + snap_cols + other_future_known_cols].copy()

pred_rec = rec.predict_recursive(history_df=train_df,
                                 horizon_dates=horizon_dates, 
                                 exogenous_future=exo_future,)
pred_rec = pred_rec.rename(columns={"yhat": "yhat_rec"})

# B. Direct — predict from the last known anchor (train_df.max())
anchor = train_df[DATE_COL].max()
pred_dir = direct.predict(history_df=train_df, anchor_date=anchor)
pred_dir = pred_dir.rename(columns={"yhat": "yhat_dir"})

# C. Quantile — feed the test design matrix
pred_q = quant.predict(test_df)

# D. Prophet — only for the top keys
prophet_dates = sorted(test_df[DATE_COL].unique())
pred_pr = prophet.predict(future_dates=prophet_dates)
pred_pr = pred_pr.rename(columns={"yhat": "yhat_pr",
                                  "yhat_lower": "yhat_pr_lo",
                                  "yhat_upper": "yhat_pr_hi"})

print("recursive:", pred_rec.shape, " direct:", pred_dir.shape,
      " quantile:", pred_q.shape, " prophet:", pred_pr.shape)


recursive: (14000, 7)  direct: (14000, 8)  quantile: (14000, 9)  prophet: (560, 9)


## 4. Score the models — point metrics

In [ ]:
from utils.metrics import metric_report

merged = (test_df
          .merge(pred_rec[[DATE_COL, *KEY_COLS, "yhat_rec"]], on=[DATE_COL, *KEY_COLS], how="left")
          .merge(pred_dir[[DATE_COL, *KEY_COLS, "yhat_dir"]], on=[DATE_COL, *KEY_COLS], how="left")
          .merge(pred_q,                                       on=[DATE_COL, *KEY_COLS], how="left")
          .merge(pred_pr,                                      on=[DATE_COL, *KEY_COLS], how="left"))

# Find the median quantile column dynamically
median_col = next(c for c in merged.columns if c.startswith("q") and abs(float(c[1:]) - 0.5) < 1e-3)
y_true = merged[TARGET_COL].values
y_train_full = train_df[TARGET_COL].values

rows = []
for label, ycol in [("LGBM-Recursive", "yhat_rec"),
                    ("LGBM-Direct",    "yhat_dir"),
                    ("LGBM-Quantile (median)", median_col),
                    ("Prophet",        "yhat_pr")]:
    yhat = merged[ycol].values
    valid_mask = ~pd.isna(yhat)
    rep = metric_report(y_true[valid_mask], yhat[valid_mask],
                        y_train=y_train_full, season=7)
    rep["model"] = label
    rep["coverage_n"] = int(valid_mask.sum())
    rows.append(rep)

metric_table = pd.DataFrame(rows)
metric_table = metric_table[["model", "MAE", "RMSE", "WAPE%", "MAPE%", "MASE", "RMSSE", "Bias", "coverage_n"]]
metric_table


### Reading the table
- **WAPE%** is the headline metric for retail-style forecasting (handles
  zeros gracefully).
- **MASE / RMSSE** normalise against the seasonal naïve baseline; values
  below 1 mean we're beating the baseline.
- **Bias** close to zero is good — large positive bias means systematic
  over-forecasting (a costly inventory issue).
- The Prophet row covers only the top-20 keys, so its `coverage_n` is
  smaller. Compare cautiously.


In [ ]:
from utils.viz import metric_bar
fig = metric_bar(metric_table, metric="WAPE%", title="Headline accuracy: WAPE% by model")
fig.show()

fig = metric_bar(metric_table, metric="RMSSE", title="RMSSE (lower=better; 1.0 = naïve)")
fig.show()


## 5. Probabilistic comparison

LightGBM-Quantile and Prophet both emit prediction bands. Score them on
calibration (coverage) and pinball loss.


In [ ]:
from utils.metrics import pinball_loss, coverage

prob_rows = []

# Quantile model
qlo = next(c for c in merged.columns if c.startswith("q") and abs(float(c[1:]) - 0.10) < 1e-3)
qhi = next(c for c in merged.columns if c.startswith("q") and abs(float(c[1:]) - 0.90) < 1e-3)
mq = ~merged[qlo].isna()
prob_rows.append({"model": "LGBM-Quantile",
                  "Pinball@10": pinball_loss(y_true[mq], merged[qlo][mq].values, q=0.10),
                  "Pinball@90": pinball_loss(y_true[mq], merged[qhi][mq].values, q=0.90),
                  "Coverage 80%": coverage(y_true[mq], merged[qlo][mq].values, merged[qhi][mq].values)})

# Prophet model
mp = ~merged["yhat_pr_lo"].isna()
prob_rows.append({"model": "Prophet",
                  "Pinball@10": pinball_loss(y_true[mp], merged["yhat_pr_lo"][mp].values, q=0.10),
                  "Pinball@90": pinball_loss(y_true[mp], merged["yhat_pr_hi"][mp].values, q=0.90),
                  "Coverage 80%": coverage(y_true[mp], merged["yhat_pr_lo"][mp].values, merged["yhat_pr_hi"][mp].values)})

prob_table = pd.DataFrame(prob_rows)
prob_table


### Calibration check
We *want* `Coverage 80%` to be ≈ 0.80. If a model says it's giving 80%
intervals but only 60% of the actuals fall inside, the model is
**over-confident**: its forecasts will appear precise but be unreliable.
That's worse than wider, well-calibrated intervals.


## 6. Visual comparison — forecast grid

Pick a few representative keys and overlay all four point forecasts against
the actual.


In [ ]:
from utils.viz import plot_forecast_grid

# `plot_forecast_grid` overlays multiple model forecasts for ONE key. We loop
# over a few representative keys and build a {model_name -> df} dict each time.
def slice_for_key(df, key_value):
    """Return rows of df where key_cols == key_value (handles tuple or scalar)."""
    if isinstance(key_value, tuple):
        return df[df.set_index(KEY_COLS).index == key_value]
    return df[df[KEY_COLS[0]] == key_value]

show_keys = list(top_keys[:4])
for k in show_keys:
    forecasts = {
        "LGBM-Recursive": slice_for_key(pred_rec, k).rename(columns={"yhat_rec": "yhat"}),
        "LGBM-Direct":    slice_for_key(pred_dir, k).rename(columns={"yhat_dir": "yhat"}),
        "LGBM-Quantile":  slice_for_key(pred_q,   k).rename(columns={median_col:  "yhat"}),
        "Prophet":        slice_for_key(pred_pr,  k).rename(columns={"yhat_pr":   "yhat"}),
    }
    fig = plot_forecast_grid(
        forecasts=forecasts,
        actual=slice_for_key(test_df, k),
        history=slice_for_key(train_df, k).tail(180),
        date_col=DATE_COL, target_col=TARGET_COL,
        title=f"Forecast comparison — key={k}",
    )
    fig.show()


## 7. Probabilistic visualisation — quantile fan + Prophet band

For one key, render the LightGBM quantile fan and Prophet's interval side by
side. The fan visually communicates "how much do we know" — a wide fan = high
uncertainty = ask for more data / bigger safety stock.


In [ ]:
from utils.viz import plot_quantile_forecast, plot_forecast

key0 = list(top_keys)[0]
mask_test_k = test_df.set_index(KEY_COLS).index == key0
mask_train_k = train_df.set_index(KEY_COLS).index == key0
mask_q = pred_q.set_index(KEY_COLS).index == key0
mask_p = pred_pr.set_index(KEY_COLS).index == key0

# Quantile fan (LightGBM)
fig_q = plot_quantile_forecast(
    quantile_df=pred_q[mask_q].drop(columns=KEY_COLS).reset_index(drop=True),
    actual=test_df[mask_test_k][[DATE_COL, TARGET_COL]],
    history=train_df[mask_train_k][[DATE_COL, TARGET_COL]].tail(180),
    date_col=DATE_COL, target_col=TARGET_COL,
    title=f"LightGBM quantile fan — key={key0}",
)
fig_q.show()

# Prophet 80% band
fig_p = plot_forecast(
    history=train_df[mask_train_k].tail(180),
    actual=test_df[mask_test_k],
    predicted=pred_pr[mask_p].rename(columns={"yhat_pr": "yhat",
                                              "yhat_pr_lo": "yhat_lower",
                                              "yhat_pr_hi": "yhat_upper"}),
    date_col=DATE_COL, target_col=TARGET_COL,
    yhat_col="yhat", yhat_lower_col="yhat_lower", yhat_upper_col="yhat_upper",
    title=f"Prophet 80% interval — key={key0}",
)
fig_p.show()


## 8. Final summary table

A single dataframe ranking all models by your business-relevant metric.


In [ ]:
final_summary = (metric_table
                 .merge(prob_table, on="model", how="left")
                 .sort_values("WAPE%"))
final_summary


## Practitioner's checklist

Before you ship a model, you should be able to say *yes* to all of these:

- [ ] **No leakage**: every feature uses lag ≥ 1 (verified in notebook 02).
- [ ] **Honest test split**: holdout dates are strictly after training dates
      (notebook 01).
- [ ] **CV-tuned hyperparameters**: you used time-aware CV folds, not
      random k-fold (notebook 05).
- [ ] **Loss matches the data**: tweedie/poisson for lumpy or
      intermittent demand; MSE for stable Gaussian-ish demand
      (notebooks 01, 03).
- [ ] **Probabilistic forecast available** for any decision involving
      stocking / staffing / capacity (notebook 06, 08).
- [ ] **Explanations ready**: gain importance + SHAP local for at least your
      top-N forecasts (notebook 07).
- [ ] **Bias near zero**: systematic over- or under-forecasting is more
      dangerous than random error (notebook 08 metric table).

Congratulations — you've built and benchmarked a production-grade
forecasting stack.
